*NLP @ HSE DSBA, autumn 2026. Author: Daria Andreeva*

# Seminar 5 — The Decoder, Alone: GPT and Where the Task Lives

Last week the body was an encoder somebody had pretrained for us, and the task lived in the head: its output shape, the point where it attaches, the axis the softmax runs along. One Linear per task, and the rest was a dial. We ended on a promise — *the other half of the transformer*: what changes when the body is a decoder and the head is the vocabulary.

Here is what changes. The head is the vocabulary, and the vocabulary is not ours to reshape — so the head never changes again. It is the same for classification, for NER, for question answering, for translation, for chat. If the task cannot live in the head, it has to live somewhere else, and the only other place is *the input*. That is the whole idea behind "prompting", two years before anyone called it that.

And it has a price. An encoder answers in one forward pass. A decoder answers one token at a time — one pass per token — and we are going to count those passes all day.

Two questions to keep in front of you. We will answer both at the end, in two lines each:

* **Why not an encoder for everything?**
* **Why not a decoder for everything?**

### The claim for today:

$$\text{solver} \;=\; \underbrace{\text{prefix}}_{\text{the task, as text}} \;\times\; \underbrace{\text{pretrained decoder}}_{\text{124M, causal}} \;\times\; \underbrace{\text{vocabulary head}}_{\text{the same for every task}}$$

The head is fixed, so the task moves into the input. What you pay for that is one forward pass per output token — and the number of output tokens is now a design decision.

# Some dependencies

In [ ]:
# Colab setup. If needed, uncomment the following lines to install dependencies and clone the repository.
# The notebook reads weeks 1 and 4 from ../, so clone the whole repo.
#!pip install -q "transformers>=5" datasets accelerate
#!git clone -q --branch autumn-2026 https://github.com/xufana/dsba_nlp.git
#%cd dsba_nlp/week05_decoder_only

In [1]:
import json
import os
import random
import re
import string
import tempfile
import time
import warnings
from collections import Counter
from dataclasses import dataclass

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from datasets import Dataset, load_dataset
import transformers
from transformers import (AutoModel, AutoModelForCausalLM, AutoTokenizer, DataCollatorForSeq2Seq, Trainer, TrainerCallback,
                          TrainingArguments)

warnings.filterwarnings("ignore")
transformers.logging.set_verbosity_error()
transformers.logging.disable_progress_bar()               # no weight-loading bars

device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
print("device:", device)

MODEL_NAME = "gpt2"                               # the body. Everything today is this one checkpoint (124M).
ENCODER_NAME = "bert-base-uncased"                # week 4's body — for one comparison in 1.1
HUB = "xufana/dsba-week05-"                       # fine-tuned copies of gpt2 live on the hub: HUB + "punct", "typos", "ner-inline", "ner-list", "qa-cq", "qa-qc"

W1, W4 = "../week01_text_representations", "../week04_pretraining_transfer"     # rows 0–2 of the results table
ART_DIR = "artifacts"
RECOMPUTE = False                                 # True: rebuild every artifact with precompute.py (~2.5 h on an M-series Mac)

N_LIVE = 2_000                                    # pairs for the one live fine-tune (3.3)
N_DEMO = 200                                      # texts behind every live measurement

/Users/andreeva.dal/personal/dsba_nlp/week05_decoder_only/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


device: mps


In [2]:
SEED = 42


def seed_all(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


def n_params(module, trainable_only=False):
    return sum(p.numel() for p in module.parameters() if p.requires_grad or not trainable_only)


IGNORE = -100                                     # the label cross_entropy skips — padding, and today: the prompt
CLASS_NAMES = ["World", "Sports", "Business", "Sci/Tech"]

# 0. Where we stopped

## 0.1 The data — AG News, once more

Same 120 000 texts, same four classes, same test set, same TF-IDF at 0.92 from week 1. Two functions we will use all day: `target_text` drops the tokens that are punctuation and nothing else (`-`, `&`) — that is the text we will ask models to restore in section 4 — and `strip_text` is what a speech recogniser hands you: lower case, no punctuation. Apostrophes stay, because they are pronounced.

In [ ]:
def target_text(text):
    """The text we ask the models to restore: the original, minus AG News' line-break and entity artifacts ("\\", "#36;")
    and minus tokens that are punctuation only ("-", "&")."""
    text = text.replace("\\", " ").replace("#36;", "$")
    return " ".join(w for w in text.split() if re.search(r"[A-Za-z0-9]", w))


def strip_text(text):
    """What ASR hands you: lower case, no punctuation. Apostrophes stay — they are pronounced."""
    return re.sub(r"[^a-z0-9' ]", "", text.lower())

In [ ]:
ds = load_dataset("fancyzhx/ag_news")
ag_train, ag_test = ds["train"], ds["test"]
demo_texts = [target_text(t) for t in ag_test["text"][:N_DEMO]]
print(demo_texts[0])
print(strip_text(demo_texts[0]))

## 0.2 Two tables

Last week had one table: AG News accuracy, one row per way of using the body. It is back, with its three rows from weeks 1 and 4, and today adds rows to it.

Today has a second table, and it is the one this seminar is about: **what an answer costs**. Four numbers per row — forward passes per item, tokens read, tokens written, and milliseconds per 1 000 items on the machine that measured it. Every reveal in section 4 writes one row into each table, and at the end we read the two side by side.

In [ ]:
results = []


def record(model, test_acc, note=""):
    results.append({"model": model, "test acc": round(test_acc, 4), "note": note})


def show_results():
    return pd.DataFrame(results).set_index("model")


week1 = pd.read_csv(f"{W1}/artifacts/reference_results.csv")
tfidf = week1[week1.representation.str.startswith("tf-idf")].sort_values("accuracy").iloc[-1]
record("TF-IDF + logreg (week 1)", tfidf.accuracy, tfidf.representation)

w4_zero = json.load(open(f"{W4}/artifacts/zero_shot.json"))["label_sets"]["v2"]
record("zero-shot through [MASK], BERT (week 4)", w4_zero["accuracy"], "body frozen, words: " + ", ".join(w4_zero["words"]))

w4_ft = json.load(open(f"{W4}/artifacts/agnews_transfer.json"))
record("fine-tuned BERT, everything (week 4)", w4_ft["runs"]["everything"]["accuracy"][-1], f"{w4_ft['n_train']:,} texts")
show_results()

In [ ]:
costs = []


def record_cost(task, body, passes, tokens_read, tokens_written, ms_per_1000, quality, note=""):
    costs.append({"task": task, "body": body, "passes / item": passes, "tokens read": tokens_read, "tokens written": tokens_written,
                  "ms / 1000 items": ms_per_1000, "quality": quality, "note": note})


def show_costs():
    return pd.DataFrame(costs).set_index(["task", "body"])

## 0.3 Three bodies, one picture

<!-- TODO: img/three_bodies.svg — encoder / encoder–decoder / decoder-only side by side: what each one reads (everything / everything, then writes / left to right), what comes out (a vector per position / a distribution over the vocabulary per position / the same), where the task lives (head / head / prefix), passes per item (1 / T_out / T_out). The same figure goes into week 3 §3.1 and week 4 §2.1. -->

Week 3 built the middle one — encoder and decoder, joined by cross-attention. Week 4 took the left one, threw the decoder away, and put a head on top. Today is the right one: the decoder from week 3 with the cross-attention removed, and *no head to put anything on*.

# 1. The decoder, alone

## 1.1 What physically comes out

Not a diagram — a shape. One sentence into BERT and into GPT-2, and we look at what each model hands back.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token           # GPT-2 has no pad token; <|endoftext|> does the job, the attention mask does the rest
gpt = AutoModelForCausalLM.from_pretrained(MODEL_NAME, attn_implementation="eager").to(device).eval()     # eager: 1.3 reads the attention weights

bert_tok = AutoTokenizer.from_pretrained(ENCODER_NAME)
bert = AutoModel.from_pretrained(ENCODER_NAME).to(device).eval()

sentence = demo_texts[0]
with torch.no_grad():
    h_bert = bert(**bert_tok(sentence, return_tensors="pt").to(device)).last_hidden_state
    logits_gpt = gpt(**tokenizer(sentence, return_tensors="pt").to(device)).logits
print("BERT :", tuple(h_bert.shape), "   a vector per token — nothing to read until a head is attached")
print("GPT-2:", tuple(logits_gpt.shape), "   a distribution over the vocabulary per token — the head is built in")

In [ ]:
blocks = n_params(gpt.transformer.h)
embeddings = n_params(gpt.transformer.wte) + n_params(gpt.transformer.wpe)
tied = gpt.lm_head.weight.data_ptr() == gpt.transformer.wte.weight.data_ptr()
print(f"GPT-2 small: {n_params(gpt):,} parameters = {embeddings:,} embeddings + {blocks:,} in {gpt.config.n_layer} blocks; head tied to the input embedding: {tied}")

Read the two lines. The encoder returns 768 numbers per token, and 768 numbers are not an answer — last week's whole point was that the answer is whatever Linear we bolt on, and its shape *is* the task. The decoder returns 50 257 numbers per token, and those *are* an answer: a probability for every token in the vocabulary. The head is already there. It is the input embedding matrix, transposed — the weight tying from week 3 §4.5 — and it accounts for close to 40M of the 124M parameters.

Which is the constraint of the day. **The task cannot go into the head, because the head is the vocabulary, and the vocabulary is not ours to change.**

One more thing before we move on — what the loss is. It is the loss from week 2, on the same axis: cross-entropy over the next token, one term per position. So the number below is a perplexity, and it reads like one.

In [ ]:
with torch.no_grad():
    enc = tokenizer(demo_texts[:64], return_tensors="pt", padding=True).to(device)
    loss = gpt(**enc, labels=enc.input_ids.masked_fill(enc.attention_mask == 0, IGNORE)).loss
print(f"cross-entropy {loss:.3f} nats per token -> perplexity {loss.exp():.1f} on {len(enc.input_ids)} news texts")

<!-- CHECK LIVE: gpt2 on AG News is roughly 40–60. -->

Do not put it next to week 2's LSTM number. Different tokenizer, different units — week 2 §2.4 is the whole story of why perplexities across tokenizers are not comparable. What *is* comparable: this is the pretrained model on news it has never seen, and it is far from lost.

## 1.2 Why twelve layers

Week 4 §4.1 asked *which layer knows the topic best* and answered with a probe per layer: a fresh logistic regression on each layer's vectors. We can ask the decoder the same question without training anything, because the decoder comes with a reader for its own residual stream — the head. Take the vector after block $k$, pass it through the final LayerNorm and the vocabulary head as if the model ended there, and read what it would have predicted. This is the *logit lens* (nostalgebraist, 2020).

Two curves, both from one forward pass: how often the layer-$k$ guess already agrees with the final one, and the perplexity of the model *truncated* at layer $k$.

**Bet:** at which layer does the top-1 guess agree with the final answer on half the positions? Draw the curve before running the cell.

In [ ]:
body, n_layers = gpt.transformer, gpt.config.n_layer
agree, nll, count = np.zeros(n_layers + 1), np.zeros(n_layers + 1), 0
with torch.no_grad():
    for i in range(0, N_DEMO, 16):
        enc = tokenizer(demo_texts[i:i + 16], return_tensors="pt", padding=True).to(device)
        hs = gpt(**enc, output_hidden_states=True).hidden_states      # 0: embeddings, k: after block k; the last one has already been through ln_f
        mask, targets = enc.attention_mask[:, 1:].bool(), enc.input_ids[:, 1:]
        final = gpt.lm_head(hs[-1])[:, :-1].argmax(-1)
        for k in range(n_layers + 1):
            logits = gpt.lm_head(hs[k] if k == n_layers else body.ln_f(hs[k]))[:, :-1]     # logits at t predict token t+1
            agree[k] += ((logits.argmax(-1) == final) & mask).sum().item()
            nll[k] += (-logits.log_softmax(-1).gather(-1, targets[..., None]).squeeze(-1) * mask).sum().item()
        count += mask.sum().item()
agree, ppl = agree / count, np.exp(nll / count)

fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
ax[0].plot(agree, marker="o"); ax[0].set(xlabel="layer", ylabel="top-1 agrees with the final layer", ylim=(0, 1.02))
ax[1].plot(ppl, marker="o"); ax[1].set(xlabel="layer", ylabel="perplexity if the model stopped here", yscale="log")
plt.tight_layout()

In [ ]:
pos = 5
enc = tokenizer(demo_texts[0], return_tensors="pt").to(device)
with torch.no_grad():
    hs = gpt(**enc, output_hidden_states=True).hidden_states
guesses = {f"layer {k}": [tokenizer.decode(t) for t in gpt.lm_head(hs[k] if k == n_layers else body.ln_f(hs[k]))[0, pos].topk(5).indices]
           for k in range(0, n_layers + 1, 2)}
print(f"prefix: {tokenizer.decode(enc.input_ids[0, :pos + 1])!r}   next token: {tokenizer.decode(enc.input_ids[0, pos + 1])!r}")
pd.DataFrame(guesses, index=[f"#{i + 1}" for i in range(5)])

<!-- CHECK LIVE: agreement is near 0 for the first layers, climbs past 0.5 around layers 8–10, the perplexity drops by orders of magnitude and keeps dropping to the last layer. Early-layer guesses are frequent function words / copies of the input; late layers commit. -->

Read the table left to right: the early layers propose *something that could follow* — a frequent word, a copy of a nearby token — and the late layers narrow it to *this* word. The perplexity curve says the same thing with a number: it drops at every layer, including the last one. There is no layer you can cut at.

❓ **Open question 1:** the probe in week 4 peaked in the *middle* of BERT — the last layers knew less about the topic than layer 7. Here the last layer is the best at every step of the curve. Both models are 12-layer transformers trained to predict tokens. What is different about *how we read them*, and why does that put the peak in a different place?

## 1.3 Where the source went

In week 3 the decoder read the source sentence through cross-attention: its queries against the *encoder's* keys and values. That layer is gone. What is left is causal self-attention: every position attends to everything on its left — and "on its left" includes the prompt.

**Bet:** a 20-token prompt, 20 generated tokens. Of the attention that generated tokens pay, what share goes to the *prompt*, and what share to the *already generated text* — by layer? Same shape everywhere, or does it change with depth?

In [ ]:
prompt = " ".join(demo_texts[1].split()[:20])
enc = tokenizer(prompt, return_tensors="pt").to(device)
with torch.no_grad():
    ids = gpt.generate(**enc, max_new_tokens=20, do_sample=False, pad_token_id=tokenizer.eos_token_id)
    attentions = gpt(ids, output_attentions=True).attentions                  # n_layers × [1, heads, T, T]
P = enc.input_ids.shape[1]
to_prompt = [a[0, :, P:, :P].sum(-1).mean().item() for a in attentions]        # generated rows, mass on prompt columns, averaged over heads and rows
print("prompt   :", prompt)
print("generated:", tokenizer.decode(ids[0, P:]))

fig, ax = plt.subplots(1, 2, figsize=(11, 3.8), gridspec_kw={"width_ratios": [1.3, 1]})
ax[0].plot(range(1, n_layers + 1), to_prompt, marker="o"); ax[0].set(xlabel="layer", ylabel="attention mass on the prompt", ylim=(0, 1))
ax[1].imshow(attentions[n_layers // 2][0].mean(0).cpu(), cmap="Blues"); ax[1].axvline(P - 0.5, color="k", lw=0.8); ax[1].axhline(P - 0.5, color="k", lw=0.8)
ax[1].set(title=f"layer {n_layers // 2 + 1}, mean over heads", xlabel="key position", ylabel="query position")
plt.tight_layout()

<!-- CHECK LIVE: the share on the prompt is high (and the first token takes a lot of it — the attention sink); the heatmap is a lower triangle with the prompt block on the left. -->

Cross-attention did not disappear — it became self-attention on the prefix. The source is just the first thing in the sequence, and the block reads it with the same weights it reads its own output with. That is the trade: one attention instead of two, one weight matrix instead of two, and one sequence for everything.

It is also a constraint, and it will come back in section 4: **the prefix is read causally**. A token of the prompt does not see what stands after it. If you put a document first and a question second, the document was encoded *without knowing the question*. Hold that thought — we will measure it.

(One more thing the decoder shares with week 3: positions. GPT-2 learns an embedding per position, like `learned` in week 3 §4.3. Almost every model after 2021 rotates queries and keys instead — RoPE — and that one is in *Try it yourself*.)

## 1.4 Three things you can do with a decoder

The body gives you $P(\text{next token} \mid \text{prefix})$ at every position, and nothing else. Everything today is one of three ways to use that:

| use | what you do | forward passes per item |
|---|---|---|
| **scorer** | compare the log-probability of a few candidate continuations | one per candidate |
| **generator** | pick tokens one at a time by a decoding rule (week 2 §4) | one per output token |
| **transformer** | fine-tune on `input => output` pairs, then generate | one per output token |

The two functions below are the whole toolkit. `generate` is greedy — when the task is in the prefix, we want the model's best answer and the same answer every time; sampling is for open text (week 2). `score` is one forward pass per pair, batched.

In [ ]:
PEAK = 0.0                  # the largest GPU reading any progress line has seen, in GB — reset per step


def gpu_gb():
    """What the GPU driver holds right now, in GB. Apple's backend has no peak counter, so we sample and keep the max."""
    global PEAK
    now = (torch.mps.driver_allocated_memory() / 1e9 if torch.backends.mps.is_available() else
           torch.cuda.max_memory_allocated() / 1e9 if torch.cuda.is_available() else 0.0)
    PEAK = max(PEAK, now)
    return now


def chunks(tag, n, size):
    """`range(0, n, size)`, plus one rewritten line with the rate, the estimate and the GPU — these loops run for minutes."""
    t0 = time.time()
    for done, i in enumerate(range(0, n, size)):
        yield i
        seen = min(i + size, n)
        if done % 10 == 9 or seen == n:
            rate = seen / max(time.time() - t0, 1e-9)
            print(f"\r  {tag} {seen:,}/{n:,}  {rate:,.0f}/s  eta {(n - seen) / rate:4.0f}s  gpu {gpu_gb():4.1f} GB   ",
                  end="", flush=True)
    print(flush=True)


@torch.no_grad()
def generate(model, tokenizer, prompts, max_new_tokens, batch_size=32):
    """Greedy, batched, left-padded. Returns the text after the prompt, up to <eos>."""
    model.eval()
    outs = []
    for i in chunks("generate", len(prompts), batch_size):
        enc = tokenizer(prompts[i:i + batch_size], return_tensors="pt", padding=True, padding_side="left").to(model.device)
        ids = model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False, pad_token_id=tokenizer.eos_token_id)
        outs += tokenizer.batch_decode(ids[:, enc.input_ids.shape[1]:], skip_special_tokens=True)
    return [o.strip() for o in outs]


@torch.no_grad()
def score(model, tokenizer, prefixes, continuations, batch_size=64):
    """log P(continuation | prefix) for every prefix × continuation, as an array [n_prefixes, n_continuations, 2]:
    the sum over the continuation's tokens and the mean per token. One forward pass per pair, batched, right-padded.

    The unembedding runs only at the positions the continuation is read from. Over the whole sequence it would be
    a [64, 262, 50257] tensor of logits — 3.4 GB, twice that with the log_softmax — and on 7 600 AG News texts
    that is what filled 14.3 GB of Apple GPU memory and pushed the machine into swap."""
    model.eval()
    body = model.transformer if hasattr(model, "transformer") else model.model
    pre = [tokenizer(p).input_ids for p in prefixes]
    con = [tokenizer(c).input_ids for c in continuations]
    pairs = [(a, b) for a in pre for b in con]
    order = sorted(range(len(pairs)), key=lambda k: len(pairs[k][0]))          # like lengths together: less padding per batch
    out = np.zeros((len(pairs), 2))
    for i in chunks("score", len(pairs), batch_size):
        chunk = [pairs[k] for k in order[i:i + batch_size]]
        seqs = [a + b for a, b in chunk]
        width = max(map(len, seqs))
        ids = torch.tensor([s + [tokenizer.eos_token_id] * (width - len(s)) for s in seqs], device=model.device)
        mask = torch.tensor([[1] * len(s) + [0] * (width - len(s)) for s in seqs], device=model.device)
        h = body(input_ids=ids, attention_mask=mask).last_hidden_state                             # [B, width, d]
        rows = torch.tensor([k for k, (a, b) in enumerate(chunk) for _ in b], device=model.device)
        cols = torch.tensor([t for a, b in chunk for t in range(len(a) - 1, len(a) + len(b) - 1)], device=model.device)
        want = torch.tensor([t for a, b in chunk for t in b], device=model.device)                  # logits at t predict token t+1
        lp = model.lm_head(h[rows, cols]).log_softmax(-1).gather(1, want[:, None]).squeeze(1).cpu().numpy()
        j = 0
        for k, (a, b) in zip(order[i:i + batch_size], chunk):
            out[k] = lp[j:j + len(b)].sum(), lp[j:j + len(b)].mean()
            j += len(b)
    return out.reshape(len(prefixes), len(continuations), 2)

In [ ]:
prefix = "The match ended after"
continuations = [" the rain", " the vote", " ninety minutes"]
print("generator:", generate(gpt, tokenizer, [prefix], max_new_tokens=12)[0])
pd.DataFrame(score(gpt, tokenizer, [prefix], continuations)[0], index=continuations, columns=["log-prob, summed", "per token"])

Decoding rules, so we do not repeat week 2: temperature scales the logits, top-k / top-p / min-p cut the tail, repetition penalties push down what was already said, beam search keeps $k$ prefixes. None of them are new today, and none of them change the *cost* — a token is a pass whatever the rule. What is new next week is the cost side: making the same generator cheaper, and making it emit only what a schema allows.

# 2. One pass, or T

Week 3 §4.6 had the sentence *"training: the whole target in one pass — that is what the causal mask buys"*. One forward pass over 64 tokens returns 64 distributions, and 64 training signals. Generating 64 tokens is 64 forward passes, because token $t$ has to exist before token $t+1$ can be predicted.

**Bet:** one pass over a 64-token prompt versus generating 64 tokens after it — how many times slower?

In [ ]:
enc = tokenizer(" ".join(demo_texts[2].split()[:48]), return_tensors="pt").to(device)
with torch.no_grad():
    t = time.time(); gpt(**enc); one_pass = time.time() - t
    t = time.time(); gpt.generate(**enc, max_new_tokens=64, min_new_tokens=64, do_sample=False, pad_token_id=tokenizer.eos_token_id); sixty_four = time.time() - t
print(f"{enc.input_ids.shape[1]} tokens in one pass: {one_pass * 1000:.0f} ms;  64 generated tokens: {sixty_four * 1000:.0f} ms  ({sixty_four / one_pass:.0f}x)")

## 2.1 What is wasted

This is `Transformer.greedy` from week 3 §4.5, as we wrote it:

```python
for _ in range(max_len):
    logits = self.decode(ys, memory, mem_mask)      # the decoder re-reads the growing prefix
    y = logits[:, -1].argmax(-1, keepdim=True)
    ys = torch.cat([ys, y], dim=1)
```

**Bet:** after $T$ steps, how many token positions has the decoder *evaluated*, in total? And how many of those did it need?

$1 + 2 + \dots + T = T(T+1)/2$, and it needed $T$. At step $t$ the only new thing is the token we just appended. Every other position's keys and values in every layer were computed at step $t-1$ and have not changed — the causal mask guarantees they cannot, because nothing on their right is allowed to reach them. So keep them. That is the **KV cache**: the model stores $K$ and $V$ per layer, and every new token adds one row to each instead of recomputing the prefix. Nothing about the maths changes; we stop repeating it.

**Bet:** `use_cache=False` against `use_cache=True`, 64 to 896 new tokens — the ratio at each length?

In [ ]:
rows = []
with torch.no_grad():
    for n in [64, 256, 512, 896]:
        for cache in [False, True]:
            t = time.time()
            gpt.generate(**enc, max_new_tokens=n, min_new_tokens=n, do_sample=False, use_cache=cache, pad_token_id=tokenizer.eos_token_id)
            rows.append({"new tokens": n, "cache": cache, "seconds": round(time.time() - t, 2)})
kv = pd.DataFrame(rows).pivot(index="new tokens", columns="cache", values="seconds")
kv["ratio"] = (kv[False] / kv[True]).round(1)
kv

<!-- CHECK LIVE: on a T4 the ratio is close to 1 at 64 tokens and grows to 3–5 by 896; on CPU it is larger. On a GPU, short generations are overhead-bound — the quadratic term only shows past a few hundred tokens. -->

Two honest remarks. On a GPU at short lengths the two are nearly equal: the model is small, the GPU is idle either way, and you are paying for launching kernels, not for arithmetic. The quadratic term is real, and it shows up when the prefix is long. And there is a bill for the cache — memory, per token, per layer — which decides how many users a GPU can serve at once. That bill, prefill against decode, and everything with a name like *TTFT* is **next week**. Today we only need the accounting below.

## 2.2 The accounting we will use today

Three numbers per item, and they are all you need to fill a card in section 3:

* **passes per item** — an encoder with a head: 1. A decoder as a scorer: one per candidate. A decoder as a generator: one per output token, $T_{out}$.
* **tokens read** — the input, once; a scorer reads it once per candidate; a generator also reads back what it wrote, one token per step (the cache makes that one, not $t$).
* **tokens written** — 0 for a head, $T_{out}$ for a generator.

The cost table's fourth column — milliseconds per 1 000 items — is measured, not derived, and the artifacts say on which machine.

In [ ]:
@dataclass
class Design:
    task: str
    body: str            # encoder / decoder / encoder-decoder
    use: str             # head / scorer / generator / transformer
    output: str          # [B,C] / [B,L,C] / [B,L,2] / a sequence over the vocabulary
    enumerable: bool     # can a head list every possible answer?
    guaranteed: str      # what the output is guaranteed to be: a label / a copy / a substring / nothing
    passes: str          # per item: "1", "candidates", "T_out"
    follow_up: str = ""  # the interviewer's push-back that flips the answer


def estimate(design, items_per_day, tokens_in, tokens_out=0, candidates=4):
    passes = {"1": 1, "candidates": candidates, "T_out": tokens_out}[design.passes]
    read = tokens_in * (candidates if design.passes == "candidates" else 1) + (tokens_out if design.passes == "T_out" else 0)
    written = tokens_out if design.passes == "T_out" else 0
    return pd.Series({"passes / day": passes * items_per_day, "tokens read / day": read * items_per_day,
                      "tokens written / day": written * items_per_day}, name=f"{design.task}: {design.body}, {design.use}")

# 3. Design first

Before we run anything, you choose. Three tasks, one card each. Then we run them both ways — a body with a head, and a decoder with the task in the prefix — and the two tables say who was right.

The card is the same every time, and every line on it is a word we already own:

```
Task / workload:        the problem as an interviewer would state it: volume, latency
Look at 20 examples:    what in the data breaks your first choice?
Output shape:           [B,C] / [B,L,C] / [B,L,2] / a sequence over the vocabulary        (week 4)
Can I enumerate it?     yes -> a head;  no -> the prefix
Guaranteed to be:       a label / a copy of the input / a substring / nothing
Body, use:              encoder, head / decoder, scorer / decoder, generator / decoder, transformer
Passes per item:        1 / candidates / T_out                                            (section 2)
Baseline to beat:       a row of the results table
Follow-up:              the push-back that flips the answer
```

Two ways to fill it, pick yours: in words (the table above, in a markdown cell), or as a `Design` and a call to `estimate` — the second one puts a number on "passes per day" before you have trained anything.

The **follow-up** line is what an interview actually sounds like. Nobody asks "encoder or decoder?". They ask you to design a system, you say encoder, and they say: *and if there are no labels? and if the categories change every week? and at ten times the volume?* Each of those flips something on the card. Write down which.

## 3.0 Worked example: classify 7 600 news texts, with no labels at all

Together, out loud. Volume: the test set. Labels: none — we are not allowed to train. Baseline to beat: TF-IDF at 0.92 (which *was* allowed to train), and week 4's zero-shot through `[MASK]` at 0.47.

First instinct: ask GPT-2. Write `This news is about` after the text, let it generate, read the word. That is the generator — one pass per output token, and then a parser for whatever it wrote, which may be *politics*, *the*, or *a*.

Look at the task again: we do not need the model to *say* anything. We need to know which of four continuations it finds most likely. That is the scorer: four forward passes, zero generated tokens, and the answer is always one of the four.

**Bet:** accuracy? Random is 0.25.

In [ ]:
prefixes = [t + " This news is about" for t in demo_texts]
gold = np.array(ag_test["label"][:N_DEMO])
class_words = [" " + c for c in CLASS_NAMES]
s = score(gpt, tokenizer, prefixes, class_words)                              # [N_DEMO, 4, 2]
pred = s[:, :, 0].argmax(1)
print(f"scorer, class names as written, summed log-prob: {(pred == gold).mean():.3f} on {N_DEMO} texts")
print("predicted as:", {CLASS_NAMES[k]: n for k, n in Counter(pred).most_common()})
print("tokens per class name:", dict(zip(CLASS_NAMES, [len(tokenizer(w).input_ids) for w in class_words])))

<!-- CHECK LIVE: with the class names as written almost everything goes to one class; " Sci/Tech" is 3–4 BPE tokens, the others 1. -->

Now the detail that is hiding in the output. `Sci/Tech` is three tokens; `World` is one. A sum of three log-probabilities is smaller than a sum of one almost by construction — so the four numbers were never comparable, and the argmax was decided by tokenization, not by the text. Two fixes, both cheap: divide by the number of tokens (a per-token score), or choose words that are one token each — *world, sports, business, technology*. The second is what week 4 did with the mask, and it is called a **verbalizer** when it has a name.

**Bet:** how much does each fix buy?

In [ ]:
if RECOMPUTE:
    !python precompute.py --only zero_shot

zs = json.load(open(f"{ART_DIR}/zero_shot_gpt2.json"))
pd.DataFrame({name: {"accuracy": r["accuracy"], "tokens per word": r["tokens_per_word"], **r["per_class"]} for name, r in zs["results"].items()}).T

In [ ]:
best = max(zs["results"], key=lambda k: zs["results"][k]["accuracy"])
record("zero-shot scorer, GPT-2, class names as written", zs["results"]["class names, sum"]["accuracy"], "4 passes per text, no training")
record(f"zero-shot scorer, GPT-2, {best}", zs["results"][best]["accuracy"], "4 passes per text, no training")
record_cost("classify", "decoder, scorer", 4, "4 × L", 0, zs["ms_per_1000"], zs["results"][best]["accuracy"], f"{zs['device']}; no training")
show_results()

<!-- REVISE AFTER PRECOMPUTE: one-token verbalizers should land somewhere around 0.5–0.6 — above week 4's mask, far below TF-IDF. -->

Below TF-IDF, and that is the right answer, not a disappointing one. The scorer guarantees the *form* of the answer — always one of four — and nothing about its *content*: a 124M model that has never seen a label does not know what we mean by *Business*. Next week does the same thing to the generator: a mask on the vocabulary at every step, so that it can only emit what a schema allows. Same idea, one level up.

The follow-up for this card: *"fine — now it is ten million texts a day."* Four passes each is forty million passes of a 124M model, and the answers are 0.55 right. Use those answers as labels, train the Linear from week 4 on them, and serve one pass of an encoder. The decoder's job was to *make the labels*, not to serve.

## 3.1 Card: restore punctuation and capitalisation

*A speech recogniser writes everything in lower case with no punctuation. One million utterances a day. Return the text as a person would have typed it.*

Fill the card. Before you choose a body, look at twenty pairs — the cell below prints them.

In [ ]:
for t in demo_texts[:20]:
    print(strip_text(t), "\n  ->", t)

Follow-ups, when you have a first answer: *and `NASA`? `U.S.`? `iPhone`? `3.5`?* — *and if the words arrive one at a time, from a live stream?*

## 3.2 Card: fix the typos in search queries

*A hundred thousand queries a minute, a tenth of them with a typo. Return the corrected query.* Twenty examples below — the noise is synthetic, six kinds of it, and the kinds are the whole point.

In [ ]:
OPS = ["sub", "del", "ins", "swap", "split", "merge"]


def add_noise(text, rate, rng):
    """Corrupt a share of the words. Returns the noisy text, one op per *gold* word ("none" if untouched), and one
    flag per *noisy* word (was it produced by an op) — the detector's labels. split and merge change the word count."""
    words, noisy, ops, flags = text.split(), [], [], []
    i = 0
    while i < len(words):
        w = words[i]
        op = rng.choice(OPS) if len(w) > 2 and rng.random() < rate else "none"
        k = rng.randrange(1, len(w) - 1) if len(w) > 2 else 0          # never the first letter — the dictionary trick needs it
        c = rng.choice(string.ascii_lowercase)
        if op == "merge" and i + 1 == len(words):
            op = "none"
        if op == "none":
            out = [w]
        elif op == "sub":
            out = [w[:k] + c + w[k + 1:]]
        elif op == "del":
            out = [w[:k] + w[k + 1:]]
        elif op == "ins":
            out = [w[:k] + c + w[k:]]
        elif op == "swap":
            out = [w[:k] + w[k + 1] + w[k] + w[k + 2:]]
        elif op == "split":
            out = [w[:k], w[k:]]
        else:
            out = [w + words[i + 1]]
            ops.append("merge")
            i += 1
        noisy += out
        ops.append(op)
        flags += [op != "none"] * len(out)
        i += 1
    return " ".join(noisy), ops, flags

In [ ]:
rng = random.Random(SEED)
for t in demo_texts[:20]:
    noisy, ops, _ = add_noise(t, 0.1, rng)
    print(noisy, "\n  ->", [f"{w} ({op})" for w, op in zip(t.split(), ops) if op != "none"])

Follow-ups: *`alot` -> `a lot`, `ca n't` -> `can't`: the number of words changes* — *and if the misspelt word is a brand?*

## 3.3 Card: extract the entities

*Named entities, four types, twenty milliseconds per document.* You built this one last week — `[B, L, 9]`, BIO tags, entity F1 of 0.89 on CoNLL-2003. The card asks you to do it *the other way*, and to say what it would cost.

Follow-ups: *`Bank of America` — an organisation with a location inside it* — *a new entity type every month* — *the client wants JSON: where do the boundaries come from?*

Fill the three cards in the cell below, in words or as `Design`s. Then start the next cell — a live fine-tune for 3.1 — and finish the cards while it runs.

In [ ]:
cards = {
    "punct": Design(task="restore punctuation and case", body=..., use=..., output=..., enumerable=..., guaranteed=..., passes=..., follow_up=...),
    "typos": Design(task="fix typos", body=..., use=..., output=..., enumerable=..., guaranteed=..., passes=..., follow_up=...),
    "ner":   Design(task="NER, 4 types", body=..., use=..., output=..., enumerable=..., guaranteed=..., passes=..., follow_up=...),
}
# estimate(cards["punct"], items_per_day=1_000_000, tokens_in=60, tokens_out=60)

The live fine-tune: 2 000 pairs `stripped => original`, one epoch, the loss only on the part after `=>` — the prompt positions are `IGNORE`, exactly the trick week 4 used for word continuations. `train` is week 4's Trainer recipe, shortened.

In [ ]:
def encode_pairs(tokenizer, prompts, targets, max_len=384):
    """`prompt target<eos>` as one sequence, the loss only on the target: prompt positions are IGNORE."""
    ids, labels = [], []
    for p, t in zip(prompts, targets):
        a = tokenizer(p).input_ids
        b = tokenizer(" " + t).input_ids + [tokenizer.eos_token_id]
        ids.append((a + b)[:max_len])
        labels.append(([IGNORE] * len(a) + b)[:max_len])
    return Dataset.from_dict({"input_ids": ids, "labels": labels})


TRAINER_DIR = os.path.join(tempfile.gettempdir(), "week05_trainer")


class EpochTimer(TrainerCallback):
    """Seconds by epoch, and one rewritten line while it trains — `disable_tqdm` silences the Trainer's own bar."""
    def on_train_begin(self, args, state, control, **kw):
        self.t0, self.seconds = time.time(), []

    def on_step_end(self, args, state, control, **kw):
        if state.global_step % 20 == 0 or state.global_step == state.max_steps:
            share = state.global_step / max(state.max_steps, 1)
            done = time.time() - self.t0
            print(f"\r  step {state.global_step:,}/{state.max_steps:,}  {share:3.0%}  {done:4.0f}s  "
                  f"eta {done / max(share, 1e-9) - done:4.0f}s  gpu {gpu_gb():4.1f} GB   ", end="", flush=True)

    def on_epoch_end(self, args, state, control, **kw):
        self.seconds.append(round(time.time() - self.t0, 1))
        print(flush=True)


def train(model, train_ds, collator, epochs=2, lr=5e-5, batch_size=16, seed=SEED, quiet=True):
    """One Trainer for every fine-tune today: AdamW, 10% linear warm-up, linear decay. Returns seconds by epoch."""
    args = TrainingArguments(output_dir=TRAINER_DIR, per_device_train_batch_size=batch_size, num_train_epochs=epochs,
                             learning_rate=lr, weight_decay=0.01, warmup_steps=0.1, lr_scheduler_type="linear",
                             logging_strategy="no", save_strategy="no", report_to=[], seed=seed, disable_tqdm=quiet,
                             dataloader_pin_memory=False)          # unsupported on MPS, and it warns once per Trainer
    timer = EpochTimer()
    trainer = Trainer(model=model, args=args, train_dataset=train_ds, data_collator=collator, callbacks=[timer])
    if quiet:
        trainer.remove_callback(transformers.PrinterCallback)
    trainer.train()
    return timer.seconds

In [ ]:
seed_all()
live = ag_train.shuffle(seed=SEED).select(range(N_LIVE))
live_targets = [target_text(t) for t in live["text"]]
live_ds = encode_pairs(tokenizer, [strip_text(t) + " =>" for t in live_targets], live_targets)
print(tokenizer.decode(live_ds[0]["input_ids"]))

punct_live = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(device)
seconds = train(punct_live, live_ds, DataCollatorForSeq2Seq(tokenizer), epochs=1, lr=5e-5, batch_size=16)
print(f"{N_LIVE:,} pairs, one epoch: {seconds[-1]:.0f}s on {device}")

## 3.4 Lightning round: which week?

Six systems, one minute each. Not "encoder or decoder" — *which week of this course solves it*, with which body, at how many passes per item.

1. Language identification, 100 000 requests per second.
2. Autocomplete in a search box, on every keystroke.
3. Spam / phishing on incoming mail.
4. Translation, fifty language pairs, server-side batches.
5. Question answering over the company's internal documents, the answer must be a quote.
6. A chat assistant, a thousand concurrent users.

<details>
<summary>Answers</summary>

1. **Week 1.** Character n-grams and a linear model; no transformer at all. The trap is reaching for a model that reads 100 000 texts a second through twelve layers.
2. **Week 2.** A trie of frequent completions, or a small n-gram / RNN language model — one step per keystroke, no prefix re-read. The trap: GPT-2 at 60 ms per keystroke.
3. **Week 1 or 4.** TF-IDF beat frozen BERT on topics; a fine-tuned encoder wins on phishing. One pass. The trap: "just ask an LLM" — one pass becomes ten, and the answer is not calibrated.
4. **Week 3.** Encoder–decoder — the source is read bidirectionally and only once; with a cache, a decoder-only reads it once too, but causally. High-resource pairs are now done both ways; the trap is "always enc-dec" *and* "always decoder".
5. **Week 4.** Two pointers, `[B, L, 2]` — the answer is a substring by construction. The trap: a generator that *usually* quotes.
6. **Week 5.** A decoder; the prefix is the conversation; the cost is $T_{out}$ per turn and the cache per user. The trap: budgeting by FLOPs and forgetting memory — next week.

</details>

# 4. The reveal

Same order as the cards. Each one: the encoder formulation and its *ceiling*, the decoder formulation and its *fidelity*, the numbers side by side, one row in each table, and who was right.

## 4.1 Punctuation and case, two ways

**The encoder way.** Token classification, as in week 4 §5.1 — one label per word, on the first piece of the word. The label has two parts: what to do with the case — `lower`, `Cap`, `CAPS` — and what to put after the word — nothing, a comma, a period, `? ! : ;`. Four cases times eight punctuations is 32 labels, and that is the encoder's *entire* vocabulary of answers.

Which raises a question we can answer before training anything. Some words are not expressible in 32 labels: `U.S.` has punctuation inside; `iPhone` and `McDonald's` are neither lower, nor Cap, nor CAPS; `Wal-Mart`, `3.5`, `(Reuters)`. They all get `other` — and `other` means *the head will get this word wrong whatever it learns*. That is the **ceiling** of the formulation, and it is worth measuring first, the way week 4's *Try it yourself* measured how much of a summary is actually in the dialogue.

**Bet:** what share of the words in AG News are beyond the 32 labels?

In [ ]:
CASES = ["lower", "Cap", "CAPS", "other"]
PUNCTS = ["", ",", ".", "?", "!", ":", ";", "other"]
PUNCT_LABELS = [f"{c}|{p}" for c in CASES for p in PUNCTS]           # 4 × 8 — the encoder's whole vocabulary of answers


def word_label(word):
    """`case|punct` for one original word. "other" on either side means the label set cannot express the word."""
    tail = re.search(r"[^A-Za-z0-9']*$", word).group()               # trailing punctuation: ",", ".", ")", '."'
    inner = word[:len(word) - len(tail)]
    punct = tail if tail in PUNCTS else "other"
    if re.sub(r"[^A-Za-z0-9']", "", inner) != inner:                 # punctuation inside or in front: U.S., Wal-Mart, (Reuters), 3.5
        punct = "other"
    letters = re.sub(r"[^A-Za-z]", "", inner)
    if letters == letters.lower():
        case = "lower"
    elif letters == letters.upper() and len(letters) > 1:
        case = "CAPS"
    elif letters == letters[:1].upper() + letters[1:].lower():
        case = "Cap"
    else:
        case = "other"                                                # iPhone, McDonald's, eBay
    return f"{case}|{punct}"


def apply_label(stripped, label):
    """The encoder's answer, rebuilt: the stripped word with the case and the trailing punctuation the label names."""
    case, punct = label.split("|")
    word = {"lower": stripped, "Cap": stripped[:1].upper() + stripped[1:], "CAPS": stripped.upper(), "other": stripped}[case]
    return word + (punct if punct != "other" else "")

In [ ]:
labels = [word_label(w) for t in demo_texts for w in t.split()]
print(f"{np.mean(['other' in l for l in labels]):.1%} of {len(labels):,} words cannot be expressed by the {len(PUNCT_LABELS)} labels")
print([w for t in demo_texts for w in t.split() if "other" in word_label(w)][:30])
pd.Series(labels).value_counts().head(10)

<!-- CHECK LIVE: around 5–8% of words; the list is abbreviations, hyphenated names, numbers with points, bracketed sources. -->

Five to eight words in a hundred, and they are not random words: they are the names, the abbreviations, the numbers — the words that carry the news. A label set can always be extended (add `CAPS.`, add `Cap-Cap`), and every extension is another class with a few training examples. At some point you are enumerating English.

**The decoder way.** No labels at all: `stripped => original`, and the model writes the original. Nothing is unexpressible. The cost is one pass per output token — and a new failure mode the encoder cannot have: the model may change a word it was only supposed to capitalise. We measure that as **changed** — the share of words whose letters came back different — and call one minus that *fidelity*.

**Bet:** word accuracy of the two, overall and on the `CAPS` and `other` words. And the decoder's `changed`?

In [ ]:
if RECOMPUTE:
    !python precompute.py --only punct

punct = json.load(open(f"{ART_DIR}/punct.json"))
enc, dec = punct["encoder"], punct["decoder"]
print(f"{punct['n_train']:,} training texts, {punct['n_test']:,} test texts ({enc['n_words']:,} words); "
      f"ceiling: {punct['ceiling']['unexpressible']:.1%} of words unexpressible;  changed by the decoder: {dec['changed']:.1%}")
print(f"word accuracy: encoder {enc['accuracy']:.3f}, decoder {dec['accuracy']:.3f}")
pd.concat({key: pd.DataFrame({"words": enc[key]["words"], "encoder": enc[key]["accuracy"], "decoder": dec[key]["accuracy"]}) for key in ["by_case", "by_punct"]})

In [ ]:
demo = punct["demo"][:5]
live_out = generate(punct_live, tokenizer, [d["input"] + " =>" for d in demo], max_new_tokens=120)
pd.DataFrame([{"gold": d["gold"], "encoder": d["encoder"], "decoder (full)": d["decoder"], f"decoder (live, {N_LIVE:,} pairs)": o} for d, o in zip(demo, live_out)]).T

In [ ]:
record_cost("punctuation + case", "encoder, 32-label head", 1, "L", 0, enc["ms_per_1000"], enc["accuracy"], f"{punct['device']}; ceiling {punct['ceiling']['unexpressible']:.1%}")
record_cost("punctuation + case", "decoder, transformer", "T_out ≈ L", "2 L", "L", dec["ms_per_1000"], dec["accuracy"], f"{punct['device']}; changed {dec['changed']:.1%}")
show_costs()

<!-- REVISE AFTER PRECOMPUTE: expected — encoder ~0.95+ overall, 0 on `other` by construction, weaker on CAPS; decoder close overall, better on CAPS/other, a few percent of words changed; decoder ms/1000 an order of magnitude above the encoder. -->

Who was right. The encoder is cheaper by a factor of about $L$ — one pass against one per word — and it *cannot* damage a word: the letters go in and the same letters come out, that is what a label is. Its price is the ceiling: on the words the labels cannot express it is wrong by construction, and those are the words that matter. The decoder has no ceiling, and it has a bill: $L$ passes, and a small share of words that came back different — a normaliser that occasionally edits.

The second follow-up decides which bill you can afford. *Words arrive one at a time, from a live stream*: the encoder reads bidirectionally, so it must wait for the phrase to end before it can label the first word — its latency is the length of the utterance. A causal model labels as the words come, with what it has; and when the next word arrives, the last comma may have to move. Same task, and the product flips the answer.

## 4.2 Typos: detect, or rewrite

**The encoder way**, honestly stated. An encoder can *point*: one label per word, `ok` / `wrong` — `[B, L, 2]`. It cannot write the right word, because the right word is not one of two labels. So the encoder detects, and something else corrects: the nearest dictionary word by edit distance, week 1's tools. Two stages, one pass.

Look at the six kinds of noise again. Four of them keep the word count — a letter substituted, dropped, inserted, two letters swapped. Two do not: a word split in two, two words merged. **Bet:** on which of the six is the encoder pipeline at zero *by construction*?

**The decoder way.** `noisy => clean`. Nothing about word counts.

In [ ]:
if RECOMPUTE:
    !python precompute.py --only typos

typos = json.load(open(f"{ART_DIR}/typos.json"))
enc, dec = typos["encoder"], typos["decoder"]
print(f"{typos['n_train']:,} training texts, {typos['n_test']:,} test texts, noise rate {typos['rate']:.0%};  "
      f"detector precision {enc['detection_precision']:.2f}, recall {enc['detection_recall']:.2f}")
pd.DataFrame({"words": enc["by_op"]["words"], "encoder + dictionary": enc["by_op"]["accuracy"], "decoder": dec["by_op"]["accuracy"]}).loc[["none"] + OPS]

In [ ]:
pd.DataFrame([{"noisy": d["input"], "gold": d["gold"], "encoder + dictionary": d["encoder"], "decoder": d["decoder"]} for d in typos["demo"][:4]]).T

In [ ]:
record_cost("typos", "encoder, detect + dictionary", 1, "L", 0, enc["ms_per_1000"], enc["accuracy"], f"{typos['device']}; split/merge unreachable")
record_cost("typos", "decoder, transformer", "T_out ≈ L", "2 L", "L", dec["ms_per_1000"], dec["accuracy"], typos["device"])
show_costs()

<!-- REVISE AFTER PRECOMPUTE: encoder pipeline: high on `none`, decent on sub/del/ins/swap, 0.0 on split and merge; decoder: lower on `none` (it edits), non-zero on split/merge. -->

Zero on `split`, zero on `merge`, and it is not a training problem — it is the formulation. A label per input word can only ever describe *that* word; it has nowhere to say "these two are one" or "this one is two". That is the boundary of the encoder way, sharper than in 4.1: not *expensive to express*, but *not expressible*. The usual escape is to extend the labels into edit operations — `$KEEP`, `$DELETE`, `$APPEND_the`, `$MERGE` — which is GECToR (Omelianchuk et al. 2020), an encoder with a vocabulary of edits; it is the same move as adding labels in 4.1, taken to its end.

And the decoder's row on `none`: it edits words that were fine. Fidelity again, and it is the price of having no ceiling.

The follow-up — *the misspelt word is a brand* — is in *Try it yourself*: run the model from 4.1 over a list of product names and count what it "corrects".

## 4.3 NER: the same CoNLL, generated

Week 4's head: `[B, L, 9]`, one BIO tag per word, and everything the model can say is a tag on a word that exists. The decoder has to *write* the entities, and there are two natural ways to write them:

* **inline** — a copy of the sentence with brackets: `[LOC Germany] 's representative to the [ORG European Union] ...`. Output length ≈ input length plus two tokens per entity.
* **list** — only the entities: `Germany: LOC; European Union: ORG`. Short — and the boundaries have to be matched back to the sentence.

Both outputs have to be parsed back to tags to score them with the same entity F1 as week 4, and the parser is where the new failure modes appear: an entity whose words are *not in the sentence* — rewritten, invented — has nowhere to go. We count those as **unmatched**. And an entity with the right words and the wrong type: **wrong type**.

**Bet:** entity F1 of the two formats against week 4's fine-tuned BERT; and the share of predicted entities that are unmatched.

In [ ]:
def bio_spans(tags):
    """[(start, end, type)] from a BIO tag sequence; end is exclusive."""
    spans, start = [], None
    for i, t in enumerate(tags + ["O"]):
        if start is not None and not t.startswith("I-"):
            spans.append((start, i, tags[start][2:]))
            start = None
        if t.startswith("B-"):
            start = i
    return spans


def to_inline(words, tags):
    """`[LOC Germany] 's representative to the [ORG European Union]` — a copy of the sentence with brackets."""
    out, i = [], 0
    for start, end, typ in bio_spans(tags):
        out += words[i:start] + [f"[{typ} {' '.join(words[start:end])}]"]
        i = end
    return " ".join(out + words[i:])


def to_list(words, tags):
    """`Germany: LOC; European Union: ORG` — only the entities, in order. `none` when there are none."""
    return "; ".join(f"{' '.join(words[s:e])}: {t}" for s, e, t in bio_spans(tags)) or "none"

In [ ]:
if RECOMPUTE:
    !python precompute.py --only ner

ner = json.load(open(f"{ART_DIR}/ner.json"))
w4_ner = json.load(open(f"{W4}/artifacts/ner_curves.json"))["runs"]["everything"]
rows = {"BERT, token head (week 4)": {"entity F1": w4_ner["entity_f1"][-1], **{t: w4_ner[f"f1_{t}"][-1] for t in ["PER", "ORG", "LOC", "MISC"]}}}
for fmt, r in ner["formats"].items():
    rows[f"GPT-2, {fmt}"] = {"entity F1": r["entity_f1"], **r["f1_by_type"], "unmatched": r["unmatched_share"], "wrong type": r["wrong_type_share"],
                             "tokens written": r["tokens_written_mean"]}
print(f"{ner['n_train']:,} training sentences, {ner['n_test']:,} test sentences")
pd.DataFrame(rows).T

In [ ]:
d = ner["formats"]["inline"]["demo"][1]
print(" ".join(d["words"]))
print("gold:     ", to_inline(d["words"], d["gold"]))
print("inline:   ", d["generated"])
print("list:     ", ner["formats"]["list"]["demo"][1]["generated"])

In [ ]:
record_cost("NER", "encoder, token head (week 4)", 1, "L", 0, None, w4_ner["entity_f1"][-1], "week 4, everything fine-tuned; not timed")
for fmt, r in ner["formats"].items():
    record_cost("NER", f"decoder, {fmt}", "T_out", "L + T_out", r["tokens_written_mean"], r["ms_per_1000"], r["entity_f1"], f"{ner['device']}; unmatched {r['unmatched_share']:.1%}")
show_costs()

<!-- REVISE AFTER PRECOMPUTE: expected — both formats below BERT on 124M; inline above list; unmatched a few percent, larger for list; wrong type comparable. -->

At 124M the generator is below the head, and the more interesting number is *how* it is wrong. The head can only miss or mislabel; it cannot invent, because a tag has to sit on a word. The generator invents — a few percent of its entities are not in the text — and it confuses types more, because the type is a word it writes rather than a column it fills. Scale changes the first number and not the second: Zhan, Wang & Huang (2026) fine-tune 8B decoders on the same CoNLL and get *above* the encoder baselines, with the same profile of errors — encoders omit, generators over-extract. Not worse; dearer, and wrong differently.

The follow-ups are where the generator earns its bill. *`Bank of America`*: BIO cannot nest — one tag per word, and `America` is either inside the organisation or a location, not both. The inline format nests for free: `[ORG Bank of [LOC America]]`. *A new type every month*: the head is a matrix with nine rows; the generator is a word in the prompt.

## 4.4 QA: answer, or point

Week 4's answer was two pointers: `[B, L, 2]`, start and end, and the answer is a substring *by construction* — there is no way for that head to say anything that is not in the paragraph. The generator writes an answer. Usually it copies; nothing forces it to.

And the bet from 1.3 comes due here. The paragraph is read causally. Put it before the question and it was encoded without knowing what to look for; put it after and every token of it can see the question. Two fine-tunes, identical except for the order of two strings in the prompt.

**Bet:** exact match against week 4's fine-tuned BERT; the share of answers that are not a substring of the paragraph; and which order wins.

In [ ]:
if RECOMPUTE:
    !python precompute.py --only qa

qa = json.load(open(f"{ART_DIR}/qa.json"))
w4_qa = json.load(open(f"{W4}/artifacts/qa_curves.json"))["runs"]["everything"]
rows = {"BERT, two pointers (week 4)": {"EM": w4_qa["exact_match"][-1], "F1": w4_qa["f1"][-1], "not a substring": 0.0}}
for order, r in qa["orders"].items():
    rows[f"GPT-2, {order}"] = {"EM": r["exact_match"], "F1": r["f1"], "not a substring": r["not_substring"], "tokens read": r["tokens_read_mean"]}
print(f"{qa['n_train']:,} training questions, {qa['n_dev']:,} dev questions")
pd.DataFrame(rows).T

In [ ]:
d = qa["orders"]["context, question"]["demo"][0]
print(d["context"][:400], "...\n")
print("Q:", d["question"], "\ngold:", d["gold"], "\ncontext first:", repr(d["pred"]), "\nquestion first:", repr(qa["orders"]["question, context"]["demo"][0]["pred"]))

In [ ]:
record_cost("extractive QA", "encoder, two pointers (week 4)", 1, "L", 0, None, w4_qa["exact_match"][-1], "week 4, everything fine-tuned; not timed; substring by construction")
for order, r in qa["orders"].items():
    record_cost("extractive QA", f"decoder, {order}", "T_ans", "L + T_ans", "T_ans", r["ms_per_1000"], r["exact_match"], f"{qa['device']}; not a substring {r['not_substring']:.1%}")
show_costs()

<!-- REVISE AFTER PRECOMPUTE: expected — both orders well below BERT's EM on 5k questions; question-first ahead of context-first; a few percent of answers not substrings. -->

Two pointers guarantee a quote and cost one pass. The generator costs a pass per answer token, and some of its answers are paraphrases — right in meaning, wrong for a system whose contract is *quote the document*. And the order matters, in the direction 1.3 predicted: the paragraph that was read *knowing the question* is the one the model answers from better. With an encoder that question does not exist — every token sees every other.

❓ **Open question 2:** week 4's Open question 2 was the unanswerable question — SQuAD 2.0 lets the two pointers *both* land on `[CLS]`, and "no answer" costs no new parameters. The generator has no `[CLS]` to point at. What would you train it to write, and what does that do to "not a substring"?

## 4.5 Both tables

In [ ]:
show_results()

In [ ]:
show_costs()

Read the cost table down the *passes* column and it splits in two: the heads at 1, the generators at $T$. Read it down the *quality* column and it does not split — on every task the head is competitive or ahead at this model size, and the generator is where the head has a ceiling. Cheaper is not worse. It is *narrower*.

# 5. Where this goes

The two questions from the top, two lines each.

**Why not an encoder for everything?** Because a head has to enumerate its answers. When the set of answers is open — variable length, insertions, nesting, a type that did not exist last month — the task has nowhere to live but the prefix, and the prefix needs a decoder.

**Why not a decoder for everything?** Because a token costs a pass, and the output is guaranteed to be nothing in particular — not a label, not a copy, not a substring, not a schema; because its probabilities are scores rather than calibrated confidences; and because it reads its input causally. Every one of those has a row in the cost table.

The course so far, in one table:

| week | body | pretraining | where the task lives | passes / item | AG News |
|---|---|---|---|---|---|
| 1 | TF-IDF, n-grams | none — counting | the head (logreg) | 1 | 0.92 |
| 1 | word2vec / fastText | skip-gram, CBOW | the head, over averaged vectors | 1 | below TF-IDF |
| 2 | RNN language model | next token | the decoding rule | $T_{out}$ | — |
| 2–3 | seq2seq LSTM ± attention | conditional next token | the pair (source, target) | $T_{out}$ | — |
| 3 | transformer encoder–decoder | conditional next token | the pair | $T_{out}$ | — |
| 4 | BERT encoder | MLM (+ NSP) | the head: shape, attachment, axis | 1 | 0.91 fine-tuned |
| 5 | GPT decoder | next token | the prefix | candidates or $T_{out}$ | <!-- REVISE --> zero-shot |

**Five things that transfer to every model this course will touch:**

1. **The head is the vocabulary, so the task is the prefix.** Prompting is not a trick; it is the only place left.
2. **One pass or $T$ — count before you choose.** The cheapest generator is the one you did not need.
3. **An encoder's ceiling is its label set — measure it before training.** The unexpressible words are the ones that matter.
4. **A generator's failure mode is doing more than it was asked.** Measure fidelity, not just accuracy.
5. **Look at twenty examples before choosing a body.** Every card in section 3 was decided by a detail in the data, not by an architecture.

**Next week:** the same decoder, cheaper. The KV cache with numbers — memory per token, prefill against decode, why a GPU runs out of memory before it runs out of arithmetic; speculative decoding — a small model drafts, the big one checks; and constrained generation — the mask on the vocabulary that makes a generator emit only what a schema allows, the generator's version of what the scorer did in 3.0.

## Try it yourself

* **Fix the product names.** Take the normaliser from 4.1 (`HUB + "punct"`) and run it over a few hundred product names and brands — `Xiaomi`, `Lenovo`, `Wal-Mart`, `eBay`, `iPod`. Count what it changes. Then design a guard: a whitelist, a rule that keeps any word whose letters were altered, a threshold on the model's own probability of the copy. Which guard costs what?
* **Streaming punctuation.** Give the encoder from 4.1 a window of $k$ words to the right, $k = 0 \dots 5$, by masking everything beyond the window, and plot word accuracy against $k$. Where does it reach the full-context number, and what latency does that $k$ mean at speech rate?
* **Headline generation.** AG News on the hub has a title for every text. Fine-tune GPT-2 on `description => title`, score with ROUGE against the baseline "the first sentence of the description". When is the baseline honest?
* **Fifty examples.** Train week 4's head on 50 labelled AG News texts; put four labelled examples into GPT-2's prefix and score the fifth. Compare accuracy — and compute what the prefix costs per text at a million texts a day. (Next week's *prefix caching* is the answer to the second number.)
* **Contrastive search.** Su et al. (2022): `gpt.generate(..., penalty_alpha=0.6, top_k=4)` against greedy and top-p on the same prompt. Look at the degeneration penalty in the formula and say why this rule is not in week 2's table.
* **RoPE.** Replace the learned positions in week 3's decoder by a rotation of the queries and keys (Su et al. 2021), train it on g2p, and test on words longer than any in training. Does the rotation extrapolate where the table did not?
* **The remaining cards.** Fill the card for each of the six lightning-round systems, in words, with a follow-up each.

## References

**GPT.** Radford, Narasimhan, Salimans & Sutskever (2018), *Improving Language Understanding by Generative Pre-Training* — GPT-1: the decoder alone, next-token pretraining, one head per task. Radford et al. (2019), *Language Models are Unsupervised Multitask Learners* — GPT-2: the task in the prompt, zero-shot.

**Reading the stack.** nostalgebraist (2020), *Interpreting GPT: the logit lens* — 1.2. Alammar (2019), *The Illustrated GPT-2* — masked self-attention and the cache, with pictures. Xiao et al. (2023), *Efficient Streaming Language Models with Attention Sinks* — why the first token takes so much attention in 1.3.

**Generation.** Holtzman et al. (2019), *The Curious Case of Neural Text Degeneration* (week 2). von Platen (2020), *How to generate text* — the Hugging Face `generate` API. Su et al. (2022), *A Contrastive Framework for Neural Text Generation* — *Try it yourself*. Pope et al. (2022), *Efficiently Scaling Transformer Inference* — the KV cache bill, next week.

**The task in the prefix.** Schick & Schütze (2021), *Exploiting Cloze Questions for Few-Shot Text Classification* — verbalizers, 3.0. Brown et al. (2020), *Language Models are Few-Shot Learners* — the prefix as the whole interface. Zhang et al. (2025), *Do BERT-Like Bidirectional Models Still Perform Better on Text Classification in the Era of LLMs?* — the head, still. Zhan, Wang & Huang (2026), *Assessment of Generative Named Entity Recognition in the Era of Large Language Models* — 4.3: formats, and how generators are wrong. Omelianchuk et al. (2020), *GECToR — Grammatical Error Correction: Tag, Not Rewrite* — 4.2: the encoder with a vocabulary of edits. Tam et al. (2024), *Let Me Speak Freely?* — what a forced format costs, next week.

**Positions.** Su et al. (2021), *RoFormer: Enhanced Transformer with Rotary Position Embedding* — *Try it yourself*.